#Assignment 1: The "Sarcasm & Sentiment" Trap

Focus: Lexicon-based vs. Machine Learning sentiment analysis in the presence of nuance.

The Task

Students are given a dataset of product reviews or tweets containing heavy sarcasm, irony, or double negatives (e.g., "Oh great, another system update that breaks my Wi-Fi. Exactly what I wanted!").

Step 1: Apply a standard lexicon-based approach (like VADER or TextBlob) to calculate sentiment scores.

Step 2: Train a supervised machine learning model (e.g., TF-IDF + Logistic Regression) on a labeled training set.

Step 3: Identify the top 20 misclassified reviews for both methods.

In [1]:
! pip install vaderSentiment
! pip install textblob
! python -m textblob.download_corpora

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 2.6 MB/s eta 0:00:00
[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package conll2000 to /root/nltk_data...
[nltk_data]   Unzipping corpora/conll2000.zip.
[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.
Finished.


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
# For Lexicon-based: pip install nltk vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


In [3]:

# 1. Mock Dataset: Highly sarcastic/nuanced reviews
data = {
    'text': [
        "Oh great, another flight delay. Exactly what I wanted!",
        "I love paying $15 for a microscopic salad. Truly fantastic.",
        "The battery life lasts a whole ten minutes. Brilliant engineering.",
        "This phone is not bad at all, actually surprisingly good.",
        "I hate how much I love this terrible reality TV show.",
        "The customer service agent was incredibly helpful and solved my issue immediately.",
        "Terrible product, broke within two minutes of opening the box.",
        "Best purchase I have made all year, highly recommend!"
    ],
    'label': [0, 0, 0, 1, 1, 1, 0, 1] # 0 = Negative, 1 = Positive
}



In [4]:
df = pd.DataFrame(data)


In [14]:

# --- PART A: Lexicon-Based Approach ---
analyzer = SentimentIntensityAnalyzer()

def get_vader_label(text):
    ### YOUR CODE HERE: Run VADER polarity scores and return 1 if compound >= 0 else 0
    score = analyzer.polarity_scores(text)
    return 1 if score['compound'] >= 0 else 0


In [16]:

df['vader_pred'] = df['text'].apply(get_vader_label)


In [17]:

# --- PART B: Machine Learning Approach ---
X_train, X_test, y_train, y_test = train_test_split(df['text'],
                                                    df['label'],
                                                    test_size=0.5, random_state=42)

# Initialize TF-IDF Vectorizer
vectorizer = TfidfVectorizer()

# Fit and transform X_train, transform X_test
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Initialize and train Logistic Regression model
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

# Make predictions
ml_preds = model.predict(X_test_tfidf)


In [18]:

vectorizer = TfidfVectorizer()

In [10]:
### YOUR CODE HERE: Fit and transform X_train, transform X_test


X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


In [11]:

model = LogisticRegression()
model.fit(X_train_tfidf, y_train)
ml_preds = model.predict(X_test_tfidf)


In [20]:

# --- PART C: Error Analysis Identification ---
print("--- VADER Misclassifications ---")
print(df[df['label'] != df['vader_pred']][['text', 'label', 'vader_pred']])


--- VADER Misclassifications ---
                                                text  label  vader_pred
0  Oh great, another flight delay. Exactly what I...      0           1
1  I love paying $15 for a microscopic salad. Tru...      0           1
2  The battery life lasts a whole ten minutes. Br...      0           1
4  I hate how much I love this terrible reality T...      1           0


In [21]:

print("\n--- ML Misclassifications ---")
# Student analysis wrapper
ml_misclassified_indices = (ml_preds != y_test).to_numpy().nonzero()[0]
ml_misclassified_reviews = X_test.iloc[ml_misclassified_indices]
ml_misclassified_labels = y_test.iloc[ml_misclassified_indices]
ml_misclassified_preds = ml_preds[ml_misclassified_indices]

# Create a DataFrame to display the misclassified results clearly
ml_misclassified_df = pd.DataFrame({
    'text': ml_misclassified_reviews,
    'label': ml_misclassified_labels,
    'ml_pred': ml_misclassified_preds
})
print(ml_misclassified_df)



--- ML Misclassifications ---
                                                text  label  ml_pred
1  I love paying $15 for a microscopic salad. Tru...      0        1
5  The customer service agent was incredibly help...      1        0
0  Oh great, another flight delay. Exactly what I...      0        1


write a short brief explaining the specific linguistic triggers (idioms, sarcasm, negation) that caused both models to fail.

Propose and implement a specific preprocessing or feature engineering step (e.g., using n-grams or dependency parsing) to fix at least some of these errors and demonstrate the before-and-after accuracy.